# Silver Order Products ETL

## Purpose
Transform and **unify** Bronze order_products (prior + train) into a single clean Silver table with composite key validation.

## Input → Output
* **Source 1:** `big_data.bronze.order_products_prior` (~32.4M rows)
* **Source 2:** `big_data.bronze.order_products_train` (~1.4M rows)
* **Target:** `big_data.silver.order_products` (~33.8M rows)
* **Composite Key:** `(order_id, product_id)`

## Transformations
1. **Load and Tag** - Load both sources, tag with dataset='prior'/'train', cast types, filter NULL FKs
2. **Union and Finalize** - Union both datasets, add timestamp, drop Bronze metadata

## Data Quality
* **Technical:** NOT NULL (composite key), UNIQUE (order_id, product_id), critical columns validation
* **Business:** Expected values (dataset), referential integrity (order_id → orders, product_id → products_enriched), reorder distribution

## Persistence
Writes to Delta table **only if all validations pass**.

### SETUP

In [0]:
%run ../UTILS/utils

In [0]:
# PySpark imports
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, BooleanType

In [0]:
# Schema configuration
source_schema = "big_data.bronze"
target_schema = "big_data.silver"

# Source tables (to be unified)
source_table_prior = "order_products_prior"
source_table_train = "order_products_train"

# Target table
target_table = "order_products"

# Composite Primary Key
primary_key_columns = ["order_id", "product_id"]

# Foreign Key columns
foreign_key_columns = ["order_id", "product_id"]

# Critical columns (NOT NULL required)
critical_columns = ["order_id", "product_id", "add_to_cart_order"]

# Expected dataset values
expected_datasets = ["prior", "train"]

# Print configuration
print("Configuration:")
print(f"  Source 1: {source_schema}.{source_table_prior}")
print(f"  Source 2: {source_schema}.{source_table_train}")
print(f"  Target: {target_schema}.{target_table}")
print(f"  Composite Key: {', '.join(primary_key_columns)}")

### TRANFORMATION

In [0]:
print("Step 1: Loading and tagging both sources...")

# Load prior and tag
prior_df = spark.table(f"{source_schema}.{source_table_prior}") \
    .withColumn("dataset", F.lit("prior")) \
    .withColumn("order_id",          F.col("order_id").cast(IntegerType())) \
    .withColumn("product_id",        F.col("product_id").cast(IntegerType())) \
    .withColumn("add_to_cart_order", F.col("add_to_cart_order").cast(IntegerType())) \
    .withColumn("reordered",         F.col("reordered").cast(BooleanType())) \
    .filter(F.col("order_id").isNotNull()) \
    .filter(F.col("product_id").isNotNull())

print(f"  Prior loaded: {prior_df.count():,} rows")

# Load train and tag
train_df = spark.table(f"{source_schema}.{source_table_train}") \
    .withColumn("dataset", F.lit("train")) \
    .withColumn("order_id",          F.col("order_id").cast(IntegerType())) \
    .withColumn("product_id",        F.col("product_id").cast(IntegerType())) \
    .withColumn("add_to_cart_order", F.col("add_to_cart_order").cast(IntegerType())) \
    .withColumn("reordered",         F.col("reordered").cast(BooleanType())) \
    .filter(F.col("order_id").isNotNull()) \
    .filter(F.col("product_id").isNotNull())

print(f"  Train loaded: {train_df.count():,} rows")
print(f"\n  Both sources tagged with 'dataset' column")

In [0]:
print("Step 2: Unifying both sources into single table...")

# Union both datasets
order_products_silver = prior_df.unionByName(train_df) \
    .withColumn("_silver_timestamp", F.current_timestamp()) \
    .drop("ingestion_timestamp", "source_file")

print(f"  Unified: {order_products_silver.count():,} rows")
print(f"  Single table with 'dataset' column (prior/train)")

print("\nPreview:")
order_products_silver.show(5, truncate=False)

# Distribution by dataset
print("\nDistribution by dataset:")
order_products_silver.groupBy("dataset").count().show()

In [0]:
# Create final DataFrame for validation and persistence
df_result = order_products_silver

print(f"\nFinal DataFrame 'df_result' created: {df_result.count():,} rows")
print("\nReady for validation and persistence")

### DATA QUALITY

In [0]:
# Execute technical validations using UTILS orchestrator
validation_technical, total_rows = technical_validations(
    df=df_result,
    primary_key_columns=primary_key_columns,
    critical_columns=critical_columns,
    range_checks=[]  # No range checks needed for this table
)

print(f"\nExpected: ~33.8M rows")

In [0]:
# Business validations
print_validation_header("Business Validations")

validation_business = True

# 1. Expected values - dataset
expected_datasets = ["prior", "train"]
actual_datasets = [row.dataset for row in df_result.select("dataset").distinct().collect()]
unexpected = set(actual_datasets) - set(expected_datasets)
if len(unexpected) > 0:
    print(f"⚠️ Unexpected dataset values: {unexpected}")
    validation_business = False
else:
    print(f"✓ dataset: All values in expected set {expected_datasets}")

# 2. Referential integrity - order_id
orders_df = spark.table(f"{target_schema}.orders")
orphans_orders = df_result.join(
    orders_df.select("order_id"),
    "order_id",
    "left_anti"
).count()
if orphans_orders > 0:
    print(f"⚠️ Found {orphans_orders:,} orphan order_id values")
    validation_business = False
else:
    print(f"✓ order_id: All values exist in orders table")

# 3. Referential integrity - product_id
products_df = spark.table(f"{target_schema}.products_enriched")
orphans_products = df_result.join(
    products_df.select("product_id"),
    "product_id",
    "left_anti"
).count()
if orphans_products > 0:
    print(f"⚠️ Found {orphans_products:,} orphan product_id values")
    validation_business = False
else:
    print(f"✓ product_id: All values exist in products_enriched table")

# 4. Reorder distribution
reordered_count = df_result.filter(F.col("reordered") == True).count()
reorder_pct = reordered_count / total_rows * 100
print(f"\nReorder Distribution:")
print(f"  Reordered: {reordered_count:,} ({reorder_pct:.1f}%)")
print(f"  First time: {total_rows - reordered_count:,} ({100-reorder_pct:.1f}%)")

print("\n" + "="*60)
if validation_business:
    print("SUCCESS: Business validations PASSED")
else:
    print("FAILURE: Business validations FAILED")
print("="*60)

In [0]:
# Combine technical and business validation results using UTILS orchestrator
validation_passed = combined_validation_result(validation_technical, validation_business)

### PERSISTENCE

In [0]:
# Conditionally persist to Delta table using UTILS function
if validation_passed:
    persist_to_delta(df_result, f"{target_schema}.{target_table}")
    
    # Additional success message
    final_count = spark.table(f"{target_schema}.{target_table}").count()
    print("\n" + "="*60)
    print("SILVER LAYER COMPLETE - All 3 tables created!")
    print("="*60)
    print(f"\nSilver Layer Summary:")
    print(f"  1. orders: {spark.table(f'{target_schema}.orders').count():,} rows")
    print(f"  2. products_enriched: {spark.table(f'{target_schema}.products_enriched').count():,} rows")
    print(f"  3. order_products: {final_count:,} rows")
    print("\nNext Step: Run Gold layer notebooks")
else:
    print("\n" + "="*60)
    print("ABORTED: Validation failed - table NOT persisted")
    print("="*60)
    print("\nFix validation errors above and re-run.")